In [1]:
!pip install -q torch transformers datasets pandas scikit-learn

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import random

## Dataset Class

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts, self.labels, self.tokenizer, self.max_length = texts, labels, tokenizer, max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

## Model Architecture

In [4]:
class MTLSarcasm(nn.Module):
    def __init__(self, num_emotions=28, num_sentiment=5, num_nli=3):
        super().__init__()
        self.trunk = AutoModel.from_pretrained('microsoft/deberta-v3-base')
        H = self.trunk.config.hidden_size
        self.sentiment_head = nn.Linear(H, num_sentiment)
        self.emotion_head = nn.Linear(H, num_emotions)
        self.nli_head = nn.Linear(H, num_nli)
        self.sarcasm_head = nn.Linear(H, 2)

    def _mean_pool(self, token_states, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (token_states * mask).sum(1) / mask.sum(1).clamp(min=1e-8)

    def forward(self, input_ids, attention_mask, task):
        tok = self.trunk(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if task == 'nli':
            return self.nli_head(tok[:, 0, :])
        pooled = self._mean_pool(tok, attention_mask)
        if task == 'sentiment':
            return self.sentiment_head(pooled)
        if task == 'emotion':
            return self.emotion_head(pooled)
        if task == 'sarcasm':
            return self.sarcasm_head(pooled)

## Device & Tokenizer

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

## Load Emotion Data

In [6]:
ds_emotion = load_dataset("google-research-datasets/go_emotions", "simplified", split='train')
emotion_labels = [x[0] if len(x) > 0 else 27 for x in ds_emotion['labels']]
emotion_data = TextDataset(ds_emotion['text'], emotion_labels, tokenizer)

README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

## Load Sentiment Data

In [7]:
ds_sentiment = load_dataset("SetFit/sst5", split='train')
sentiment_data = TextDataset(ds_sentiment['text'], ds_sentiment['label'], tokenizer)

README.md:   0%|          | 0.00/421 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: 0.00B [00:00, ?B/s]

dev.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/8544 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1101 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2210 [00:00<?, ? examples/s]

## Load NLI Data

In [8]:
ds_mnli = load_dataset("nyu-mll/multi_nli", split='train').filter(lambda x: x['label'] != -1)
ds_anli = load_dataset("facebook/anli")
ds_anli_train = concatenate_datasets([ds_anli['train_r1'], ds_anli['train_r2'], ds_anli['train_r3']])
random.seed(42)
mnli_idx = random.sample(range(len(ds_mnli)), min(15000, len(ds_mnli)))
mnli_texts = [f"{ds_mnli[i]['premise']} {tokenizer.sep_token} {ds_mnli[i]['hypothesis']}" for i in mnli_idx]
mnli_labels = [ds_mnli[i]['label'] for i in mnli_idx]
anli_texts = [f"{p} {tokenizer.sep_token} {h}" for p, h in zip(ds_anli_train['premise'], ds_anli_train['hypothesis'])]
anli_labels = list(ds_anli_train['label'])
nli_data = TextDataset(mnli_texts + anli_texts, mnli_labels + anli_labels, tokenizer)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/4.94M [00:00<?, ?B/s]

data/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/5.10M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

...

Generating dev_r2 split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test_r2 split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_r3 split:   0%|          | 0/100459 [00:00<?, ? examples/s]

Generating dev_r3 split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test_r3 split:   0%|          | 0/1200 [00:00<?, ? examples/s]

## Load Sarcasm Data

In [9]:
all_sarc_texts, all_sarc_labels = [], []
ds_headlines = load_dataset("raquiba/Sarcasm_News_Headline", split='train')
all_sarc_texts.extend(list(ds_headlines['headline']))
all_sarc_labels.extend(list(ds_headlines['is_sarcastic']))
ds_semeval22 = load_dataset("csv", data_files="https://raw.githubusercontent.com/iabufarha/iSarcasmEval/main/train/train.En.csv", split='train')
for _ in range(5):
    all_sarc_texts.extend(list(ds_semeval22['tweet']))
    all_sarc_labels.extend(list(ds_semeval22['sarcastic']))
ds_semeval18 = load_dataset("cardiffnlp/tweet_eval", "irony", split='train')
all_sarc_texts.extend(list(ds_semeval18['text']))
all_sarc_labels.extend(list(ds_semeval18['label']))
ds_reddit = load_dataset("marcbishara/sarcasm-on-reddit", split='sft_train')
reddit_idx = random.sample(range(len(ds_reddit)), min(10000, len(ds_reddit)))
all_sarc_texts.extend([ds_reddit[i]['comment'] for i in reddit_idx])
all_sarc_labels.extend([ds_reddit[i]['label'] for i in reddit_idx])
sarcasm_data = TextDataset(all_sarc_texts, all_sarc_labels, tokenizer)

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/28619 [00:00<?, ? examples/s]

...

Generating sft_validation split:   0%|          | 0/30325 [00:00<?, ? examples/s]

Generating reward_train split:   0%|          | 0/272922 [00:00<?, ? examples/s]

Generating reward_validation split:   0%|          | 0/30325 [00:00<?, ? examples/s]

Generating ppo_train split:   0%|          | 0/272924 [00:00<?, ? examples/s]

Generating ppo_validation split:   0%|          | 0/30325 [00:00<?, ? examples/s]

## Initialize Model

In [10]:
model = MTLSarcasm().to(device)
for param in model.parameters():
    if param.dtype == torch.float16:
        param.data = param.data.float()

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

## DataLoaders

In [11]:
emotion_loader = DataLoader(emotion_data, batch_size=24, shuffle=True, num_workers=2, pin_memory=True)
sentiment_loader = DataLoader(sentiment_data, batch_size=24, shuffle=True, num_workers=2, pin_memory=True)
nli_loader = DataLoader(nli_data, batch_size=24, shuffle=True, num_workers=2, pin_memory=True)
sarcasm_loader = DataLoader(sarcasm_data, batch_size=24, shuffle=True, num_workers=2, pin_memory=True)

## Training Setup

In [12]:
model.trunk.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

ce_loss = nn.CrossEntropyLoss()
sarcasm_weights = torch.tensor([1.0, 2.0]).to(device)
ce_sarcasm = nn.CrossEntropyLoss(weight=sarcasm_weights)

optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3
total_steps = len(sarcasm_loader) * epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = torch.amp.GradScaler('cuda')

task_weights = {'sarcasm': 1.0, 'sentiment': 0.8, 'emotion': 0.5, 'nli': 0.5}

## Training Loop

In [13]:
def _cycle(iterator, loader):
    batch = next(iterator, None)
    if batch is None:
        iterator = iter(loader)
        batch = next(iterator)
    return batch, iterator

model.train()
for epoch in range(epochs):
    iter_emotion = iter(emotion_loader)
    iter_sentiment = iter(sentiment_loader)
    iter_nli = iter(nli_loader)
    running = {k: 0. for k in ['emotion', 'sentiment', 'nli', 'sarcasm', 'total']}
    log_interval = 100
    for step, batch_sarc in enumerate(sarcasm_loader):
        optimizer.zero_grad(set_to_none=True)
        batch_emo, iter_emotion = _cycle(iter_emotion, emotion_loader)
        batch_sent, iter_sentiment = _cycle(iter_sentiment, sentiment_loader)
        batch_nli, iter_nli = _cycle(iter_nli, nli_loader)
        with torch.amp.autocast('cuda'):
            loss_emo = ce_loss(model(batch_emo['input_ids'].to(device), batch_emo['attention_mask'].to(device), 'emotion'), batch_emo['labels'].to(device))
            loss_sent = ce_loss(model(batch_sent['input_ids'].to(device), batch_sent['attention_mask'].to(device), 'sentiment'), batch_sent['labels'].to(device))
            loss_nli = ce_loss(model(batch_nli['input_ids'].to(device), batch_nli['attention_mask'].to(device), 'nli'), batch_nli['labels'].to(device))
            loss_sarc = ce_sarcasm(model(batch_sarc['input_ids'].to(device), batch_sarc['attention_mask'].to(device), 'sarcasm'), batch_sarc['labels'].to(device))
            total_loss = (loss_sarc * task_weights['sarcasm'] + loss_sent * task_weights['sentiment'] + loss_emo * task_weights['emotion'] + loss_nli * task_weights['nli'])
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running['emotion'] += loss_emo.item()
        running['sentiment'] += loss_sent.item()
        running['nli'] += loss_nli.item()
        running['sarcasm'] += loss_sarc.item()
        running['total'] += total_loss.item()
        if (step + 1) % log_interval == 0:
            avg = {k: v / log_interval for k, v in running.items()}
            print(f"Ep {epoch+1}/{epochs} | Stp {step+1}/{len(sarcasm_loader)} | Emo: {avg['emotion']:.4f} | Sent: {avg['sentiment']:.4f} | NLI: {avg['nli']:.4f} | Sarc: {avg['sarcasm']:.4f} | Tot: {avg['total']:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
            running = {k: 0. for k in running}
    print(f"--- Epoch {epoch+1}/{epochs} complete ---")

/tmp/ipykernel_57/1502894236.py:31: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Ep 1/3 | Stp 100/2451 | Emo: 3.4245 | Sent: 1.6257 | NLI: 1.2135 | Sarc: 0.7105 | Tot: 4.3301 | LR: 2.72e-06
Ep 1/3 | Stp 200/2451 | Emo: 2.8920 | Sent: 1.5832 | NLI: 1.1764 | Sarc: 0.6715 | Tot: 3.9723 | LR: 5.44e-06
Ep 1/3 | Stp 300/2451 | Emo: 2.7371 | Sent: 1.5757 | NLI: 1.1356 | Sarc: 0.6575 | Tot: 3.8543 | LR: 8.16e-06
Ep 1/3 | Stp 400/2451 | Emo: 2.7246 | Sent: 1.5365 | NLI: 1.1252 | Sarc: 0.6407 | Tot: 3.7948 | LR: 1.09e-05
Ep 1/3 | Stp 500/2451 | Emo: 2.6080 | Sent: 1.4645 | NLI: 1.1118 | Sarc: 0.5990 | Tot: 3.6305 | LR: 1.36e-05
Ep 1/3 | Stp 600/2451 | Emo: 2.5694 | Sent: 1.3905 | NLI: 1.0911 | Sarc: 0.6098 | Tot: 3.5524 | LR: 1.63e-05
Ep 1/3 | Stp 700/2451 | Emo: 2.5298 | Sent: 1.3703 | NLI: 1.0646 | Sarc: 0.5969 | Tot: 3.4904 | LR: 1.90e-05
Ep 1/3 | Stp 800/2451 | Emo: 2.3919 | Sent: 1.2982 | NLI: 0.9844 | Sarc: 0.5835 | Tot: 3.3103 | LR: 1.98e-05
Ep 1/3 | Stp 900/2451 | Emo: 2.3328 | Sent: 1.2580 | NLI: 0.8179 | Sarc: 0.5640 | Tot: 3.1458 | LR: 1.95e-05
Ep 1/3 | Stp 1000/2

## Save Model

In [14]:
torch.save(model.state_dict(), "mtl_sarcasm.pth")

## Benchmark: iSarcasmEval

In [21]:
model.eval()
ds_test_22 = load_dataset("csv", data_files="https://raw.githubusercontent.com/iabufarha/iSarcasmEval/main/test/task_A_En_test.csv", split='train')
test_22_data = TextDataset(ds_test_22['text'], ds_test_22['sarcastic'], tokenizer)
test_22_loader = DataLoader(test_22_data, batch_size=32, shuffle=False)
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_22_loader:
        with torch.amp.autocast('cuda'):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), 'sarcasm')
        probs = torch.softmax(logits, dim=1)
        preds = (probs[:, 1] > 0.8).int().cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].tolist())
print("=== iSarcasmEval ===")
print(f"F1-sarcastic: {f1_score(all_labels, all_preds, average='binary', pos_label=1):.4f}")
print(classification_report(all_labels, all_preds, target_names=['Not Sarcastic', 'Sarcastic']))
print(confusion_matrix(all_labels, all_preds))

=== iSarcasmEval ===
F1-sarcastic: 0.4017
               precision    recall  f1-score   support

Not Sarcastic       0.91      0.86      0.88      1200
    Sarcastic       0.35      0.47      0.40       200

     accuracy                           0.80      1400
    macro avg       0.63      0.66      0.64      1400
 weighted avg       0.83      0.80      0.81      1400

[[1030  170]
 [ 107   93]]


## Benchmark: News Headlines

In [28]:
ds_headlines_test = load_dataset("raquiba/Sarcasm_News_Headline", split='test')
test_hl_data = TextDataset(ds_headlines_test['headline'], ds_headlines_test['is_sarcastic'], tokenizer)
test_hl_loader = DataLoader(test_hl_data, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_hl_loader:
        with torch.amp.autocast('cuda'):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), 'sarcasm')
        preds = torch.argmax(logits, dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].tolist())
print("=== News Headlines ===")
print(f"F1-sarcastic: {f1_score(all_labels, all_preds, average='binary', pos_label=1):.4f}")
print(classification_report(all_labels, all_preds, target_names=['Not Sarcastic', 'Sarcastic']))
print(confusion_matrix(all_labels, all_preds))

Repo card metadata block was not found. Setting CardData to empty.


=== News Headlines ===
F1-sarcastic: 0.9253
               precision    recall  f1-score   support

Not Sarcastic       0.96      0.92      0.94     14985
    Sarcastic       0.90      0.95      0.93     11724

     accuracy                           0.93     26709
    macro avg       0.93      0.93      0.93     26709
 weighted avg       0.93      0.93      0.93     26709

[[13756  1229]
 [  571 11153]]


## Benchmark: TweetEval Irony

In [26]:
ds_test_18 = load_dataset("cardiffnlp/tweet_eval", "irony", split='test')
test_18_data = TextDataset(ds_test_18['text'], ds_test_18['label'], tokenizer)
test_18_loader = DataLoader(test_18_data, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_18_loader:
        with torch.amp.autocast('cuda'):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), 'sarcasm')
        probs = torch.softmax(logits, dim=1)
        preds = (probs[:, 1] > 0.5).int().cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].tolist())
print("=== TweetEval Irony ===")
print(f"F1-sarcastic: {f1_score(all_labels, all_preds, average='binary', pos_label=1):.4f}")
print(classification_report(all_labels, all_preds, target_names=['Not Sarcastic', 'Sarcastic']))
print(confusion_matrix(all_labels, all_preds))

=== TweetEval Irony ===
F1-sarcastic: 0.6118
               precision    recall  f1-score   support

Not Sarcastic       0.76      0.59      0.66       473
    Sarcastic       0.53      0.72      0.61       311

     accuracy                           0.64       784
    macro avg       0.65      0.65      0.64       784
 weighted avg       0.67      0.64      0.64       784

[[278 195]
 [ 88 223]]
